I Think the best lr/epoch combo would be 0.001 and 500

I want to test different lr schedulers

In [27]:
# Add project root to path so we can import from data_manipulation and model
import sys
from pathlib import Path

def _find_project_root():
    cwd = Path.cwd()
    if (cwd / "data_manipulation").is_dir():
        return cwd
    if (cwd.parent / "data_manipulation").is_dir():
        return cwd.parent
    return cwd  # fallback

project_root = _find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [28]:
import torch
from torch import nn
import matplotlib.pyplot as plt
from functools import partial
import numpy as np


# Imports from data_manipulation and model
from data_manipulation.data_split import create_dataloader, DemandDataset
from model.functions import pinball_loss, rmse, train, get_test_loss, pinball_loss_tensor, pinball_loss_sum

In [29]:
class thread_net(nn.Module):

    @staticmethod
    def _mlp(sizes: list[int]) -> nn.Sequential:
        """Linear layers with ReLU between; no ReLU after the last linear."""
        if len(sizes) < 2:
            raise ValueError("layer_list needs at least [input_dim, output_dim]")
        layers: list[nn.Module] = []
        for i in range(len(sizes) - 1):
            layers.append(nn.Linear(sizes[i], sizes[i + 1]))
            if i < len(sizes) - 2:
                layers.append(nn.ReLU())
        return nn.Sequential(*layers)

    def __init__(self, layer_list=[12, 40, 40, 40, 1], num_nets=500):
        super().__init__()
        self.layer_list = list(layer_list)
        self.nets = nn.ModuleList(
            [self._mlp(self.layer_list) for _ in range(num_nets)]
        )
        self.num_nets = num_nets

    def forward(self, x):
        outs = [net(x[:, i::self.num_nets]) for i, net in enumerate(self.nets)]
        return torch.cat(outs, dim=1)

In [30]:
all_specs = [
    "7_day_rolling_ema",	
    "7_day_rolling_mean",	
    "30_day_rolling_ema",	
    "diff_180_day",	
    "diff_90_day",	
    "30_day_rolling_mean",	
    "diff_30_day",	
    "7_day_rolling_min",	
    "7_day_lag",	
    "30_day_rolling_min",	
    "14_day_lag",	
    "diff_365_day",	
]

In [31]:
# Get data for spec
train_loader, val_loader, test_loader = create_dataloader(
    batch_size=8, 
    test_batch_size=1,
    pin_memory=False,
    specs=all_specs,
    combine_items=True,
    combine_stores=True
    )

In [32]:
# Parameters
h_cost = 1
l_cost = 3

num_nets = train_loader.dataset.y.shape[1]
in_per_net = train_loader.dataset.x.shape[1] // num_nets
layer_list = [in_per_net, 40, 40, 40, 1]  

In [33]:
base_lr = 0.001
epochs = 500
eval_interval = 20
num_trials_per_config = 3  # repeat each scheduler config; leaderboard uses mean avg test loss


def train_with_scheduler(
    net,
    optimizer,
    scheduler,
    scheduler_name,
    scheduler_step_mode,
    loss_fn,
    train_loader,
    val_loader,
    epochs,
    eval_interval,
    device="cpu",
):
    """Train loop that supports both step-wise and plateau schedulers."""
    train_losses = []
    val_losses = []
    train_loader_iter = iter(train_loader)

    print(f"\n[{scheduler_name}] Starting training for {epochs} steps")
    for step in range(epochs):
        try:
            x, y = next(train_loader_iter)
        except StopIteration:
            train_loader_iter = iter(train_loader)
            x, y = next(train_loader_iter)

        x, y = x.to(device), y.to(device)

        net.train()
        optimizer.zero_grad()
        y_hat = net(x)
        train_loss = loss_fn(y_hat, y)
        train_loss.backward()
        optimizer.step()

        if scheduler_step_mode == "batch":
            scheduler.step()

        train_losses.append(train_loss.item())

        should_eval = (step + 1) % eval_interval == 0 or step == 0 or step == (epochs - 1)
        if should_eval:
            net.eval()
            val_loss_sum = 0.0
            val_batches = 0
            with torch.no_grad():
                for x_val, y_val in val_loader:
                    x_val, y_val = x_val.to(device), y_val.to(device)
                    y_val_hat = net(x_val)
                    val_loss_sum += loss_fn(y_val_hat, y_val).item()
                    val_batches += 1
            mean_val_loss = val_loss_sum / val_batches if val_batches else 0.0
            val_losses.append(mean_val_loss)

            if scheduler_step_mode == "plateau":
                scheduler.step(mean_val_loss)

            current_lr = optimizer.param_groups[0]["lr"]
            print(
                f"[{scheduler_name}] Step {step + 1:>4}/{epochs} | "
                f"train_loss={train_loss.item():.4f} | "
                f"val_loss={mean_val_loss:.4f} | "
                f"lr={current_lr:.8f}"
            )

    return train_losses, val_losses


scheduler_configs = [
    # Top 3 from prior single-run leaderboard — re-run with num_trials_per_config
    {
        "name": "OneCycleLR_pct0.2_div10",
        "build": lambda optimizer: (
            torch.optim.lr_scheduler.OneCycleLR(
                optimizer,
                max_lr=base_lr,
                total_steps=epochs,
                pct_start=0.2,
                anneal_strategy="cos",
                div_factor=10.0,
                final_div_factor=1e4,
            ),
            "batch",
        ),
    },
    {
        "name": "OneCycleLR_pct0.1_div25",
        "build": lambda optimizer: (
            torch.optim.lr_scheduler.OneCycleLR(
                optimizer,
                max_lr=base_lr,
                total_steps=epochs,
                pct_start=0.1,
                anneal_strategy="cos",
                div_factor=25.0,
                final_div_factor=1e4,
            ),
            "batch",
        ),
    },
    {
        "name": "CosineAnnealingLR_tmax100_eta1e-4",
        "build": lambda optimizer: (
            torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer,
                T_max=100,
                eta_min=1e-4,
            ),
            "batch",
        ),
    },
]


In [34]:
scheduler_results = []

for scheduler_cfg in scheduler_configs:
    scheduler_name = scheduler_cfg["name"]

    print("\n" + "=" * 80)
    print(f"Running scheduler test: {scheduler_name} ({num_trials_per_config} trials)")
    print(f"Base LR: {base_lr} | Epochs: {epochs}")
    print("=" * 80)

    trial_avg_test_losses = []
    trial_final_lrs = []

    for trial in range(num_trials_per_config):
        seed = 42 + trial
        torch.manual_seed(seed)
        np.random.seed(seed)

        log_name = f"{scheduler_name} trial {trial + 1}/{num_trials_per_config}"

        net = thread_net(layer_list=layer_list, num_nets=num_nets)
        loss = partial(pinball_loss_sum, h_cost=h_cost, l_cost=l_cost)
        optimizer = torch.optim.Adam(net.parameters(), lr=base_lr)

        scheduler, scheduler_step_mode = scheduler_cfg["build"](optimizer)

        train_losses, val_losses = train_with_scheduler(
            net=net,
            optimizer=optimizer,
            scheduler=scheduler,
            scheduler_name=log_name,
            scheduler_step_mode=scheduler_step_mode,
            loss_fn=loss,
            train_loader=train_loader,
            val_loader=val_loader,
            epochs=epochs,
            eval_interval=eval_interval,
            device="cpu",
        )

        test_loss = get_test_loss(net, test_loader, loss, "cpu")
        average_test_loss = (torch.tensor(test_loss).mean() / 500).item()
        final_lr = optimizer.param_groups[0]["lr"]

        trial_avg_test_losses.append(average_test_loss)
        trial_final_lrs.append(final_lr)

        print(f"[{log_name}] Completed | avg test loss: {average_test_loss:.6f} | final_lr: {final_lr:.8f}")

    mean_avg_test_loss = float(np.mean(trial_avg_test_losses))
    std_avg_test_loss = float(np.std(trial_avg_test_losses, ddof=0))
    mean_final_lr = float(np.mean(trial_final_lrs))

    scheduler_results.append(
        {
            "scheduler": scheduler_name,
            "avg_test_loss": mean_avg_test_loss,
            "std_avg_test_loss": std_avg_test_loss,
            "trial_avg_test_losses": trial_avg_test_losses,
            "trial_final_lrs": trial_final_lrs,
            "mean_final_lr": mean_final_lr,
            "last_train_loss": train_losses[-1] if train_losses else None,
            "last_val_loss": val_losses[-1] if val_losses else None,
        }
    )

    print(f"[{scheduler_name}] All trials done | mean avg test loss: {mean_avg_test_loss:.6f} ± {std_avg_test_loss:.6f}")


Running scheduler test: OneCycleLR_pct0.2_div10 (3 trials)
Base LR: 0.001 | Epochs: 500

[OneCycleLR_pct0.2_div10 trial 1/3] Starting training for 500 steps
[OneCycleLR_pct0.2_div10 trial 1/3] Step    1/500 | train_loss=695068.1875 | val_loss=633951.0724 | lr=0.00010023
[OneCycleLR_pct0.2_div10 trial 1/3] Step   20/500 | train_loss=596046.9375 | val_loss=617731.9145 | lr=0.00018763
[OneCycleLR_pct0.2_div10 trial 1/3] Step   40/500 | train_loss=483684.5938 | val_loss=576863.0559 | lr=0.00041639
[OneCycleLR_pct0.2_div10 trial 1/3] Step   60/500 | train_loss=356552.6875 | val_loss=412259.4737 | lr=0.00069718
[OneCycleLR_pct0.2_div10 trial 1/3] Step   80/500 | train_loss=66290.6484 | val_loss=62358.3127 | lr=0.00092065
[OneCycleLR_pct0.2_div10 trial 1/3] Step  100/500 | train_loss=42433.3398 | val_loss=37536.0617 | lr=0.00099998
[OneCycleLR_pct0.2_div10 trial 1/3] Step  120/500 | train_loss=30505.0781 | val_loss=33043.9268 | lr=0.00099321
[OneCycleLR_pct0.2_div10 trial 1/3] Step  140/500 

In [35]:
print("\n" + "#" * 80)
print(
    f"Scheduler leaderboard (lowest mean avg test loss over {num_trials_per_config} trials per config)"
)
print("#" * 80)

sorted_results = sorted(scheduler_results, key=lambda x: x["avg_test_loss"])

for rank, result in enumerate(sorted_results, start=1):
    trials_str = ", ".join(f"{x:.6f}" for x in result["trial_avg_test_losses"])
    print(
        f"{rank}. {result['scheduler']:<40} | "
        f"mean_avg_test_loss={result['avg_test_loss']:.6f} | "
        f"std={result['std_avg_test_loss']:.6f} | "
        f"mean_final_lr={result['mean_final_lr']:.8f}"
    )
    print(f"    per-trial avg test loss: [{trials_str}]")

best = sorted_results[0]
print("\nBest scheduler (by mean avg test loss across trials):")
print(
    f"{best['scheduler']} with mean_avg_test_loss={best['avg_test_loss']:.6f} "
    f"(± {best['std_avg_test_loss']:.6f})"
)


################################################################################
Scheduler leaderboard (lowest mean avg test loss over 3 trials per config)
################################################################################
1. OneCycleLR_pct0.2_div10                  | mean_avg_test_loss=7.097329 | std=0.005328 | mean_final_lr=0.00000003
    per-trial avg test loss: [7.103801, 7.097435, 7.090751]
2. OneCycleLR_pct0.1_div25                  | mean_avg_test_loss=7.097621 | std=0.003884 | mean_final_lr=0.00000002
    per-trial avg test loss: [7.101005, 7.099675, 7.092182]
3. CosineAnnealingLR_tmax100_eta1e-4        | mean_avg_test_loss=7.120059 | std=0.051376 | mean_final_lr=0.00010000
    per-trial avg test loss: [7.191473, 7.072767, 7.095937]

Best scheduler (by mean avg test loss across trials):
OneCycleLR_pct0.2_div10 with mean_avg_test_loss=7.097329 (± 0.005328)
